# Milestone 1 — Train Walking Skill (revised)

**Project**: PBT Meta-Controller (Hierarchical PBT, Phase 1)
**Goal**: Train Go2 walking PPO policy in Genesis with **wide command range** so VLM can drive it later

**Strategy**: ฝึก policy เดียวที่ track command (vx, vy, wz) ใน range กว้าง → meta layer (M3+) จะ dispatch command เหล่านี้

---

## Lessons learned (incorporated in this revision)

Compared to first attempt, this notebook fixes 4 things we learned the hard way:

1. **`rsl-rl-lib==5.0.0`** (not 2.2.4) — required because Genesis main branch's `examples/locomotion` is now TensorDict-native
2. **Reweight `tracking_ang_vel: 0.2 → 1.0`** — default weight is too low when ang_vel_range is widened. With 0.2, policy learns to ignore turning entirely
3. **250 iter** (not 101) — wider command space needs more PPO iterations
4. **All env operations wrapped in `torch.inference_mode()`** — Genesis env tensors become inference tensors after training; writing to them outside inference_mode raises RuntimeError

---

## Prerequisites

- M0 PASS (Genesis + Ollama installed)
- Same Colab session as M0 (Genesis already initialized)
- Or: fresh session, but expect longer first build

**Time budget**: ~40 min (build 1 min + train 35 min + eval 1 min)

## Step 1 — Pre-flight check

In [ ]:
# Step 1: GPU memory baseline + verify Genesis available
import subprocess
import torch

print("=" * 60)
print("STEP 1: Pre-flight check")
print("=" * 60)

# GPU memory
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True
)
used, total = [int(x.strip()) for x in result.stdout.strip().split(",")]
free = total - used
print(f"GPU memory: {used/1024:.1f} / {total/1024:.1f} GB used ({free/1024:.1f} GB free)")

if free / 1024 >= 12:
    print("[OK] >= 12 GB free — plenty for 4096 envs (uses ~1 GB)")
elif free / 1024 >= 6:
    print("[OK] >= 6 GB free — fine for 4096 envs")
else:
    print("[WARN] < 6 GB free — kill other processes first")

# PyTorch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Genesis (must be installed from M0)
try:
    import genesis as gs
    v = gs.__version__ if hasattr(gs, '__version__') else 'unknown'
    print(f"Genesis: {v}")
except ImportError:
    print("[FAIL] Genesis not installed — run M0 first")
    raise

## Step 2 — Initialize Genesis (idempotent)

In [ ]:
# Step 2: Initialize Genesis if not already initialized
import genesis as gs

try:
    gs.init(backend=gs.gpu, precision="32", logging_level="warning")
    print("[OK] Genesis initialized fresh")
except Exception as e:
    msg = str(e).lower()
    if "already" in msg or "initialized" in msg:
        print("[OK] Genesis already initialized (from M0 in this session)")
    else:
        print(f"[FAIL] Unexpected: {e}")
        raise

## Step 3 — Install rsl-rl 5.0.0

**Critical**: must be 5.0.0, not 2.2.4. Genesis `examples/locomotion` uses TensorDict-native API which 2.2.4 doesn't understand.

In [ ]:
# Step 3: Install rsl-rl 5.0.0 + tensorboard
import subprocess

print("Installing rsl-rl-lib==5.0.0 + tensorboard...")
result = subprocess.run(
    ["pip", "install", "-q", "rsl-rl-lib==5.0.0", "tensorboard"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("[OK] rsl-rl 5.0.0 installed")
    import rsl_rl
    print(f"  loaded from {rsl_rl.__file__}")
else:
    print(f"[FAIL] {result.stderr[-500:]}")

## Step 4 — Download Genesis Go2 example files

We download `go2_env.py`, `go2_train.py`, `go2_eval.py` directly from GitHub. These ship with Genesis as examples but aren't installed by pip.

In [ ]:
# Step 4: Download Genesis Go2 example files
import os
import urllib.request

GENESIS_BASE = "https://raw.githubusercontent.com/Genesis-Embodied-AI/Genesis/main/examples/locomotion"
FILES = ["go2_env.py", "go2_train.py", "go2_eval.py"]

WORK_DIR = "/content/go2_train"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print("Downloading Genesis Go2 example files...")
for fname in FILES:
    url = f"{GENESIS_BASE}/{fname}"
    try:
        urllib.request.urlretrieve(url, fname)
        size = os.path.getsize(fname)
        print(f"  [OK] {fname} ({size} bytes)")
    except Exception as e:
        print(f"  [FAIL] {fname}: {e}")

print(f"\nFiles in {WORK_DIR}:")
for f in sorted(os.listdir(".")):
    print(f"  {f}")

## Step 5 — Patch get_cfgs (command range + reward weights)

Two changes to default Genesis config:

1. **Wide command range**: vx ∈ [-1, 1], vy ∈ [-0.5, 0.5], wz ∈ [-1, 1]
   (default is vx=0.5 only, vy=wz=0)
2. **Reweight `tracking_ang_vel`**: 0.2 → 1.0
   (default 0.2 made sense when wz_range=0, but with wz∈[-1,1] policy ignores turning unless we boost the gradient)

We patch via monkey-patch (don't edit go2_train.py source) so the original file stays clean.

In [ ]:
# Step 5: Patch get_cfgs
import sys
import importlib

sys.path.insert(0, "/content/go2_train")

import go2_train
importlib.reload(go2_train)  # ensure fresh state if cell re-run

# Save original
original_get_cfgs = go2_train.get_cfgs

# Show original for comparison
env_cfg, obs_cfg, reward_cfg, command_cfg = original_get_cfgs()
print("=== ORIGINAL command_cfg ===")
for k, v in command_cfg.items():
    print(f"  {k}: {v}")
print()
print("=== ORIGINAL reward_scales ===")
for k, v in reward_cfg["reward_scales"].items():
    print(f"  {k}: {v}")
print()

# Define patched version
def patched_get_cfgs():
    env_cfg, obs_cfg, reward_cfg, command_cfg = original_get_cfgs()

    # 1. Wide command range
    command_cfg["lin_vel_x_range"] = [-1.0, 1.0]
    command_cfg["lin_vel_y_range"] = [-0.5, 0.5]
    command_cfg["ang_vel_range"]   = [-1.0, 1.0]

    # 2. Reweight ang_vel tracking from 0.2 → 1.0 (parity with lin_vel)
    reward_cfg["reward_scales"]["tracking_ang_vel"] = 1.0

    return env_cfg, obs_cfg, reward_cfg, command_cfg

go2_train.get_cfgs = patched_get_cfgs

# Verify + assertions
env_cfg, obs_cfg, reward_cfg, command_cfg = go2_train.get_cfgs()
print("=== PATCHED command_cfg ===")
for k, v in command_cfg.items():
    marker = "  ★" if "range" in k else "   "
    print(f"{marker} {k}: {v}")
print()
print("=== PATCHED reward_scales ===")
for k, v in reward_cfg["reward_scales"].items():
    marker = "  ★" if k == "tracking_ang_vel" else "   "
    print(f"{marker} {k}: {v}")

# Hard assertions — fail loud if patch didn't take
assert command_cfg["lin_vel_x_range"] == [-1.0, 1.0], "vx range patch failed"
assert command_cfg["lin_vel_y_range"] == [-0.5, 0.5], "vy range patch failed"
assert command_cfg["ang_vel_range"]   == [-1.0, 1.0], "wz range patch failed"
assert reward_cfg["reward_scales"]["tracking_ang_vel"] == 1.0, "ang_vel reweight failed"
assert reward_cfg["reward_scales"]["tracking_lin_vel"] == 1.0, "lin_vel changed unexpectedly"
print("\n[OK] All assertions passed — patches verified")

## Step 6 — Build environment with 4096 envs

In [ ]:
# Step 6: Build env
import torch
import gc
import time

print("=" * 60)
print("STEP 6: Build env (4096 envs)")
print("=" * 60)

# Clean up any prior env from previous notebook runs
try:
    del env
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Memory before
free_before, total = torch.cuda.mem_get_info()
print(f"Before build: {free_before/1e9:.1f} / {total/1e9:.1f} GB free")
print()

NUM_ENVS = 4096

from go2_env import Go2Env

env_cfg, obs_cfg, reward_cfg, command_cfg = go2_train.get_cfgs()

print(f"Building Go2Env with num_envs={NUM_ENVS}...")
t0 = time.time()
env = Go2Env(
    num_envs=NUM_ENVS,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    show_viewer=False,
)
elapsed = time.time() - t0

free_after, _ = torch.cuda.mem_get_info()
used_by_env = (free_before - free_after) / 1e9
print(f"[OK] Build succeeded in {elapsed:.1f}s")
print(f"  env consumed: {used_by_env:.2f} GB")
print(f"  free after build: {free_after/1e9:.1f} GB")

# Sanity: verify env actually uses our reweighted scale
if hasattr(env, 'reward_scales'):
    actual_ang = env.reward_scales.get('tracking_ang_vel', 'NOT FOUND')
    print(f"\nenv.reward_scales['tracking_ang_vel'] = {actual_ang}")
    assert actual_ang == 1.0 * env.dt if isinstance(actual_ang, float) and actual_ang != 1.0 else True
    # Note: Genesis may multiply scales by dt internally — that's OK as long as it's not zero

## Step 7 — Inspect env interface (TensorDict structure)

Genesis main branch is **TensorDict-native** — `env.reset()` returns a TensorDict, not a raw tensor. We verify the structure here so M2 (fast loop) knows what to expect.

In [ ]:
# Step 7: Inspect env interface
import torch

print("=" * 60)
print("STEP 7: Env interface inspection")
print("=" * 60)

# IMPORTANT: env tensors are inference tensors after build → wrap in inference_mode
with torch.inference_mode():
    reset_res = env.reset()

    print(f"env.reset() type: {type(reset_res).__name__}")

    if hasattr(reset_res, 'keys'):
        print(f"  keys: {list(reset_res.keys())}")
        if hasattr(reset_res, 'batch_size'):
            print(f"  batch_size: {reset_res.batch_size}")
        for k in reset_res.keys():
            v = reset_res[k]
            if hasattr(v, 'shape'):
                print(f"  '{k}': shape={tuple(v.shape)} dtype={v.dtype}")

    print()
    print(f"env.num_envs: {env.num_envs}")
    print(f"env.num_actions: {env.num_actions}")
    if hasattr(env, 'obs_buf'):
        print(f"env.obs_buf shape: {tuple(env.obs_buf.shape)}")

# Expected output:
#   env.reset() type: TensorDict
#     keys: ['policy']
#     batch_size: torch.Size([4096])
#     'policy': shape=(4096, 45) dtype=torch.float32
#   env.num_envs: 4096
#   env.num_actions: 12
#   env.obs_buf shape: (4096, 45)

## Step 8 — Train PPO (250 iterations)

This is the long step (~35 min on L4). Watch for:
- `Mean total reward` should rise from ~0 toward positive
- Both `tracking_lin_vel` AND `tracking_ang_vel` reward terms should rise
- If `tracking_ang_vel` stays flat after iter 100 → reward weighting still wrong

If reward is flat after 50 iter → abort and tell Claude in chat.

In [ ]:
# Step 8: Run PPO training
import os
import time
from rsl_rl.runners import OnPolicyRunner

print("=" * 60)
print("STEP 8: PPO training (250 iter)")
print("=" * 60)

train_cfg = go2_train.get_train_cfg("go2_walking_meta_phase1")

MAX_ITER = 250
log_dir = "/content/go2_train/logs/go2_walking_meta_phase1"

# Clean stale log dir if exists
import shutil
if os.path.exists(log_dir):
    print(f"Removing stale log dir: {log_dir}")
    shutil.rmtree(log_dir)
os.makedirs(log_dir, exist_ok=True)

print(f"max_iterations: {MAX_ITER}")
print(f"num_envs: {env.num_envs}")
print(f"log_dir: {log_dir}")
print()

runner = OnPolicyRunner(env, train_cfg, log_dir, device="cuda:0")

print("Starting training (~35 min on L4)...")
print("Watch: Mean reward should rise; tracking_ang_vel should rise too")
print()

t0 = time.time()
try:
    runner.learn(num_learning_iterations=MAX_ITER, init_at_random_ep_len=True)
    elapsed = time.time() - t0
    print()
    print(f"[OK] Training complete in {elapsed/60:.1f} min ({elapsed/MAX_ITER:.1f} sec/iter)")
except KeyboardInterrupt:
    elapsed = time.time() - t0
    print(f"\n[INTERRUPTED] at {elapsed/60:.1f} min — saving checkpoint")
    runner.save(os.path.join(log_dir, "model_interrupted.pt"))
except Exception as e:
    import traceback
    print(f"[FAIL] {type(e).__name__}: {e}")
    traceback.print_exc()

## Step 9 — Inspect training reward curve

In [ ]:
# Step 9: Read tensorboard log to inspect reward curve
import os
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

print("=" * 60)
print("STEP 9: Reward curve inspection")
print("=" * 60)

log_dir = "/content/go2_train/logs/go2_walking_meta_phase1"

try:
    ea = EventAccumulator(log_dir)
    ea.Reload()
    tags = ea.Tags()['scalars']
    print(f"Available scalars ({len(tags)}):")
    for t in tags[:30]:
        print(f"  {t}")
    print()

    # Look for reward-related tags
    interesting = [t for t in tags if any(k in t.lower() for k in ['reward', 'return', 'tracking'])]

    print(f"=== Reward-related scalars ===")
    for tag in interesting:
        events = ea.Scalars(tag)
        if not events:
            continue
        print(f"\n{tag} ({len(events)} points):")
        # Sample at iter 0, 25%, 50%, 75%, 100%
        n = len(events)
        for pct in [0, 25, 50, 75, 100]:
            i = min(int(n * pct / 100), n - 1)
            print(f"  iter {events[i].step:4d}: {events[i].value:+.4f}")

except Exception as e:
    print(f"[WARN] Could not read tensorboard: {e}")
    print("Falling back to checkpoint listing only")

## Step 10 — List checkpoints

In [ ]:
# Step 10: List saved checkpoints
import os

log_dir = "/content/go2_train/logs/go2_walking_meta_phase1"
print(f"Checkpoints in {log_dir}:")

if os.path.exists(log_dir):
    pts = sorted([f for f in os.listdir(log_dir) if f.endswith(".pt")])
    for pt in pts:
        size_mb = os.path.getsize(os.path.join(log_dir, pt)) / 1e6
        print(f"  {pt}  ({size_mb:.1f} MB)")

    if pts:
        latest = pts[-1]
        print(f"\nLatest: {latest}")
else:
    print(f"[WARN] No log dir at {log_dir}")

## Step 11 — Eval: 7 commands

Test the trained policy on 7 different commands. Robot should respond differently to each.

**All env operations wrapped in `torch.inference_mode()`** because env tensors are now inference tensors after training.

In [ ]:
# Step 11: Evaluate policy on 7 test commands
import torch

print("=" * 60)
print("STEP 11: Eval — 7 commands")
print("=" * 60)

policy = runner.get_inference_policy(device="cuda:0")

with torch.inference_mode():
    obs_td = env.reset()
    print(f"Reset OK, type={type(obs_td).__name__}")
    print()

    test_commands = [
        (0.0,  0.0,  0.0,  "STOP"),
        (0.5,  0.0,  0.0,  "FORWARD"),
        (-0.3, 0.0,  0.0,  "BACKWARD"),
        (0.0,  0.0,  0.5,  "TURN_LEFT"),
        (0.0,  0.0, -0.5,  "TURN_RIGHT"),
        (0.3,  0.0,  0.5,  "FORWARD+TURN"),
        (0.0,  0.3,  0.0,  "SIDESTEP"),
    ]
    STEPS = 100  # 2 seconds @ 50 Hz

    print(f"{'cmd':14s} {'target':18s} {'vx':>8s} {'vy':>8s} {'ω_z':>8s} {'h':>6s} {'fall%':>6s}")
    print("-" * 75)

    for vx, vy, wz, label in test_commands:
        # Set commands for all envs
        env.commands[:, 0] = vx
        env.commands[:, 1] = vy
        env.commands[:, 2] = wz

        # Snapshot env 0 state before
        pos_before = env.base_pos[0].clone()

        # Run policy for STEPS steps
        for _ in range(STEPS):
            actions = policy(obs_td)
            step_res = env.step(actions)
            # step may return TensorDict directly or tuple — handle both
            if isinstance(step_res, tuple):
                obs_td = step_res[0]
            else:
                obs_td = step_res

        # Compute results
        pos_after = env.base_pos[0]
        delta = (pos_after - pos_before).cpu().numpy()
        speed_x = delta[0] / (STEPS * 0.02)
        speed_y = delta[1] / (STEPS * 0.02)

        # Yaw rate from base_ang_vel directly (signed, more reliable than quaternion delta)
        actual_wz = env.base_ang_vel[:, 2].mean().item()

        height = pos_after[2].item()
        fall_pct = (env.base_pos[:, 2] < 0.2).float().mean().item() * 100

        target_str = f"({vx:+.1f},{vy:+.1f},{wz:+.1f})"
        print(f"  {label:12s} {target_str:18s} {speed_x:+8.2f} {speed_y:+8.2f} {actual_wz:+8.2f} {height:6.2f} {fall_pct:5.0f}%")

print()
print("PASS criteria:")
print("  STOP:        |vx|, |vy|, |ω_z| < 0.05")
print("  FORWARD:     vx > +0.30 (60% of 0.5)")
print("  BACKWARD:    vx < -0.18 (60% of 0.3)")
print("  TURN_LEFT:   ω_z > +0.25 (50% of 0.5)  ← critical")
print("  TURN_RIGHT:  ω_z < -0.25 (50% of 0.5)  ← critical")
print("  FORWARD+TURN: vx > +0.15 AND ω_z > +0.20")
print("  SIDESTEP:    vy > +0.15")
print("  All: fall < 10%")

## Step 12 — Save checkpoint to Google Drive (optional)

In [ ]:
# Step 12: Save final checkpoint to Google Drive
# Uncomment below to save

# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil, os
# src = "/content/go2_train/logs/go2_walking_meta_phase1/model_250.pt"
# dst_dir = "/content/drive/MyDrive/PBT_Robotics_Hierarchical"
# os.makedirs(dst_dir, exist_ok=True)
# dst = os.path.join(dst_dir, "go2_walking_meta_phase1_v2.pt")
# shutil.copy(src, dst)
# print(f"Saved: {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

print("To save to Drive: uncomment cell above and run")

## Step 13 — Final summary

In [ ]:
# Step 13: Final memory + status
import torch
import subprocess

print("=" * 60)
print("M1 SUMMARY")
print("=" * 60)

free, total = torch.cuda.mem_get_info()
print(f"GPU memory: {(total-free)/1e9:.1f} / {total/1e9:.1f} GB used ({free/1e9:.1f} GB free)")

# Ollama still alive?
result = subprocess.run(
    ["curl", "-s", "http://localhost:11434/api/version"],
    capture_output=True, text=True, timeout=2
)
if result.returncode == 0 and result.stdout:
    print(f"Ollama service: ALIVE — {result.stdout.strip()}")
else:
    print("Ollama service: DOWN")
    print("To restart for M3: nohup ollama serve > /tmp/ollama.log 2>&1 &")

print()
print("M1 deliverables:")
print(f"  - Trained policy in /content/go2_train/logs/go2_walking_meta_phase1/")
print(f"  - 7-command eval results (Step 11)")
print(f"  - Reward curve (Step 9)")
print()
print("If turning passes criteria → ready for M2 (Fast Loop + manual command)")
print("If turning still fails → paste eval output to chat for diagnosis")

## What to report back

1. **Step 6** — env build OK? GPU memory consumed?
2. **Step 7** — TensorDict structure confirmed (key='policy', shape (4096, 45))?
3. **Step 8** — total training time + any errors during learn()
4. **Step 9** — reward curve at iter 0/62/125/187/250 for at least:
   - `Mean reward`
   - `Episode reward / tracking_lin_vel`
   - `Episode reward / tracking_ang_vel`  ← key check
5. **Step 11** — full 7-command eval table

If turning still fails after all this → next investigation is curriculum (start narrow, widen) or PPO hyperparameters (learning rate, batch size).